# Module 3: Text as Data

## Are tech companies backing away from their climate pledges?

Between 2019 and 2021, the largest tech companies made some of the most ambitious corporate climate pledges
ever announced. Google said it had been carbon neutral since 2007 and would run on carbon-free energy every
hour of every day by 2030. Microsoft promised to be carbon negative by 2030. Amazon co-founded The Climate
Pledge (net zero by 2040), Meta promised net zero across its value chain by 2030, and Apple promised a
carbon neutral business and products by 2030.

Since then, the energy demands of AI have grown faster than almost anyone expected. Journalists have
reported pieces of what happened next. In this module you will build your own answer from the companies' own
words: 37 annual environmental reports, about 3,200 pages, published between 2018 and 2026.

## The question

Are these companies backing away from their climate pledges? How has the way they write about their goals
changed, and how does your answer compare with independent assessments?

How you make this question precise is up to you, and it is the main thing you will be graded on. What
counts as "backing away"? Dropping a goal outright, narrowing what it covers, calling it a "moonshot",
admitting it has become harder, or simply no longer mentioning it?

## Data

- **The reports.** `reports.csv` lists every report with its URL. Run `bash download_data.sh` in the terminal
  to download them into `data/reports/` (about 500 MB). Companies name their reports differently: Google's
  "2024 Environmental Report" describes 2023. The `data_year` column gives the year each report describes.
- **Independent sources**, also downloaded by the script into `data/validation/`:
  - the [Science Based Targets initiative](https://sciencebasedtargets.org/target-dashboard) company export,
    which records whether each company's targets are validated or its commitments removed (today's status
    only), and
  - NewClimate Institute's [Corporate Climate Responsibility Monitor 2025](https://newclimate.org/), tech
    sector chapter, an expert assessment of all five companies.

## Tools

You have used language models in chat windows. This module adds a second kind of model, and uses both from
code.

- **A decision model** reads a text and answers *typed questions* with probabilities. It never writes text.
  We use [Jev](https://openrouter.ai/docs/guides/community/jev). It answers in a fraction of a second, and
  reading every paragraph of every report costs about 30 cents. It offers three kinds of question:
  - `noul`: a yes/no question. Jev returns the probability of "yes".
  - `choice`: a question with a list of labels you define. Jev returns the most likely label and a
    probability for each.
  - `score`: a question with an ordered list of levels you define. Jev returns a position on that scale.
- **A language model** reads a text and writes an answer. Asked to fill in a schema (a template of named
  fields), it can turn a paragraph into a structured record. It is slower and costs more than Jev. We use
  `deepseek/deepseek-v4-flash`.

Both are reached through [OpenRouter](https://openrouter.ai) with the same key you use for your coding agent.
The cells below set up both and show what each can do on a few paragraphs from the reports.

## Setup

This cell reads your key from the `OPENROUTER_API_KEY` environment variable. Never paste the key into the
notebook.

In [ ]:
import os
import time

import pandas as pd
import pymupdf
import requests
from openai import OpenAI
from pydantic import BaseModel, Field

OPENROUTER_KEY = os.environ["OPENROUTER_API_KEY"]
llm = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_KEY, timeout=90)


def decide(text, questions):
    """Ask Jev one or more typed questions about a text. Returns Jev's answers."""
    response = requests.post(
        "https://openrouter.ai/api/v1/systemone",
        headers={"Authorization": f"Bearer {OPENROUTER_KEY}"},
        json={"model": "~typesafe/jev-latest", "state": text, "questions": questions},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["answers"]

## What a PDF looks like to a computer

Page 41 of Google's 2024 report, as text. Look at where the lines break.

In [ ]:
page = pymupdf.open("data/reports/google/google-2024.pdf")[40]
print(page.get_text()[:1500])

## Six paragraphs

Read these before running anything. They are excerpts from the reports, lightly trimmed.

In [ ]:
examples = {
    "google 2024": "We aim to neutralize our residual emissions with high-quality carbon removal credits by 2030, and to do so in a way that maximizes our positive impact on global decarbonization. This approach represents an evolution of our strategy: starting in 2023, we’re no longer maintaining operational carbon neutrality. We’re instead focusing on accelerating an array of carbon solutions and partnerships that will help us work toward our net-zero goal.",
    "meta 2020": "In 2020, we announced our most ambitious goal yet: Reach net zero greenhouse gas (GHG) emissions across our value chain in 2030, aligning with the Science Based Targets initiative (SBTi) to set clear targets to help limit global temperature rise to no more than 1.5 degrees Celsius above preindustrial levels.",
    "microsoft 2025": "As we remain focused on sustained progress towards our 2030 goals, it has become clear that our journey towards being carbon negative is a marathon, not a sprint. While our total emissions (Scope 1, 2, and 3) have increased by 23.4% compared to our 2020 baseline due to growth-related factors such as AI and cloud expansion, we are encouraged by the fact that this increase has been modest compared to the 168% increase in energy use and 71% revenue growth that has taken place over the same period.",
    "google 2025": "We made progress toward our climate moonshots, which require broader change. While we remain committed to our climate moonshots, it’s become clear that achieving them is now more complex and challenging across every level. Even though we’ve successfully reduced our data center energy emissions, supply chain emissions have risen.",
    "amazon 2022": "As we examined our work toward The Climate Pledge, we realized that it no longer made sense to have a separate and more narrow Shipment Zero goal that applied to only one part of our business, so we’ve decided to eliminate it. We remain focused on The Climate Pledge and our goal to reach net-zero carbon across our operations by 2040.",
    "apple 2022": "At Apple, we’re committed to doing our part to fight climate change. For us, this means taking bold action. Our goal to be carbon neutral by 2030 is both ambitious and necessary. The efforts that support this goal require innovations at scale — like designing and implementing new technologies, mobilizing innovative financing, and rapidly deploying renewable energy.",
}
for name, text in examples.items():
    print(f"{name}:\n{text}\n")

## A yes/no question (`noul`)

Before running the cell, decide what you would answer for each paragraph.

In [ ]:
dropped = {"type": "noul",
           "instructions": "Does this paragraph say the company has dropped a climate goal or claim it previously made?"}

pd.Series({name: decide(text, {"dropped": dropped})["dropped"]["noul"] for name, text in examples.items()},
          name="P(yes)").round(2)

## The same idea, worded differently

Change the wording and run it again. Which paragraphs move, and why?

In [ ]:
weakened = {"type": "noul",
            "instructions": "Does this paragraph say the company has dropped, narrowed, delayed or stopped a climate goal or claim it previously made?"}
harder = {"type": "noul",
          "instructions": "Does this paragraph say a climate goal has become harder to reach?"}

pd.DataFrame({name: {q: a["noul"] for q, a in decide(text, {"weakened": weakened, "harder": harder}).items()}
              for name, text in examples.items()}).T.round(2)

## A category (`choice`)

You define the labels and describe each one. Jev picks one and gives a probability for every label.

In [ ]:
role = {"type": "choice",
        "instructions": "What is this paragraph mainly doing with respect to the company's climate goals?",
        "criteria": {"announce": "announces or restates a goal",
                     "progress": "reports measured progress toward a goal",
                     "setback": "admits emissions rose or that a goal is harder to reach",
                     "revise": "changes, narrows, reframes or drops a goal"}}

answer = decide(examples["microsoft 2025"], {"role": role})["role"]
print(answer["choice"])
answer["probabilities"]

## A scale (`score`)

You define the levels, in order. Jev returns a number between 0 and the last level. A value like 1.6 is a
probability-weighted average, so look at the probabilities too.

In [ ]:
tone = {"type": "score",
        "instructions": "How confident does the company sound about reaching its climate goals?",
        "criteria": ["says the goal is at risk or being abandoned",
                     "acknowledges serious challenges",
                     "neutral or matter of fact",
                     "confident and on track"]}

pd.Series({name: decide(text, {"tone": tone})["tone"]["score"] for name, text in examples.items()},
          name="tone (0-3)").round(2)

## Many questions at once

All the questions can go in one request. Note the time and cost: the corpus has about 12,000 paragraphs.

In [ ]:
start = time.time()
response = requests.post(
    "https://openrouter.ai/api/v1/systemone",
    headers={"Authorization": f"Bearer {OPENROUTER_KEY}"},
    json={"model": "~typesafe/jev-latest", "state": examples["google 2025"],
          "questions": {"weakened": weakened, "harder": harder, "role": role, "tone": tone}},
    timeout=120,
).json()
print(f"{time.time() - start:.2f} s, ${response['usage']['cost']:.6f}")
response["answers"]

## A language model fills in a record

A schema names the fields you want. The `evidence` field asks for a quote, which you can check against the
text.

In [ ]:
class Goal(BaseModel):
    goal: str = Field(description="the climate goal, in a few words")
    target_year: int | None = Field(None, description="year the goal should be met, if stated")
    status: str = Field(description="what the paragraph says is happening to the goal")
    evidence: str = Field(description="an exact quote from the paragraph supporting this record")


class Goals(BaseModel):
    goals: list[Goal]


start = time.time()
response = llm.chat.completions.parse(
    model="deepseek/deepseek-v4-flash", temperature=0, max_tokens=4000, response_format=Goals,
    extra_body={"reasoning": {"enabled": False}},
    messages=[{"role": "system", "content": "Extract every climate goal or claim the paragraph discusses, including any it says were dropped or changed. Use only the paragraph."},
              {"role": "user", "content": examples["amazon 2022"]}],
)
print(f"{time.time() - start:.1f} s")
for goal in response.choices[0].message.parsed.goals:
    print(goal)

---

# Your analysis

The rest of the notebook is yours. Your coding agent will write most of the code. What it cannot do for you
is decide what you are measuring. The written answers below are where you decide, in your own words, and
they are what you give your agent.

Keep your conversations with your agent: export them into the `prompts/` folder as you go.

Your session has limited memory. Save results that cost time or money to files in `output/`, and have your
code reuse them.

## The corpus

Which reports, and which parts of them, will you code? Every paragraph, or only some? How will you turn
the PDFs into paragraphs, and how will you check that worked?

**Your answer:**

## Your codebook

Write the questions you will ask of each paragraph, in plain English, precisely enough that your agent
could turn each into a request to Jev without asking you anything. For each question give:

- a short name,
- its type (`noul`, `choice` or `score`) and why that type,
- the exact wording,
- for `choice`, each label and its description; for `score`, each level in order,
- what you expect it to show about the companies.

Use more than one type of question. Ask at least one idea in two different ways (for example a single
`score`, and several narrow `noul` questions), so you can compare them. Expect to revise this section after
you see the results, and keep the earlier version.

**Your codebook:**

## Checking the codes

Before looking at Jev's answers, label a sample of paragraphs yourself. Then compare. Where you and Jev
disagree, read the paragraph again and decide who was right. What do the disagreements tell you about your
questions?

**Your answer:**

## Results

What do the codes show for each company over time? Each figure or table needs a sentence saying what it
shows. Say how you turned paragraph-level answers into one number per report (a count above a threshold,
a sum of probabilities, a share of paragraphs, or something else) and why.

**Your answer:**

## Records and independent sources

Codes show where to look. For the paragraphs that matter most, you might extract structured records with
the language model, and check every quote. How do your findings compare with SBTi and NewClimate
Institute? Before explaining any difference, say what each source measures.

**Your answer:**

## Conclusion

In your own words: is each company backing away from its climate pledges? How confident are you, and what
would you do next?

**Your answer:**